# Qwen2.5-7B-Instruct — MLX variants summary

Loads the FP16 / 4-bit / 2-bit result JSONs from `data/`, reconstructs the `name` + `variant` fields from each run's label, and renders the KPI table + writes `summary.csv` via `bench_utils.print_results_table` and `bench_utils.dump_results_table`.

In [6]:
import json
import logging
import re
import sys
from pathlib import Path

# Make the repo root importable so we can pull the summary helpers.
REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bench_utils import dump_results_table, print_results_table

# print_results_table logs via the `bench` logger — wire it to stdout so the
# table shows up in the notebook cell output.
bench_log = logging.getLogger("bench")
bench_log.setLevel(logging.INFO)
if not bench_log.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(message)s"))
    bench_log.addHandler(h)

DATA_DIR = Path.cwd() / "data"
print("data dir:", DATA_DIR)

data dir: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/data


In [7]:
# Label format from benchmark_mlx.py is `f"{variant} ({name})"`.
LABEL_RE = re.compile(r"^(?P<variant>.+?)\s*\((?P<name>[^)]+)\)\s*$")

# Preferred display order — FP16 first, then ascending bitwidth.
VARIANT_ORDER = {"FP16": 0, "MLX 4-bit": 1, "MLX 2-bit": 2}

def load_run(path: Path) -> dict:
    payload = json.loads(path.read_text())
    m = LABEL_RE.match(payload["label"])
    if not m:
        raise ValueError(f"unparseable label in {path.name}: {payload['label']!r}")
    row = dict(payload["results"])
    row["name"] = m.group("name")
    row["variant"] = m.group("variant")
    return row

rows = [load_run(p) for p in sorted(DATA_DIR.glob("*.json"))]
rows.sort(key=lambda r: (r["name"], VARIANT_ORDER.get(r["variant"], 99)))

for r in rows:
    print(f"  {r['variant']:<10}  {r['name']}")

  MLX 4-bit   Gemma-2-9B-it
  MLX 4-bit   Llama-3.1-8B-Instruct
  FP16        Qwen2.5-7B-Instruct
  MLX 4-bit   Qwen2.5-7B-Instruct
  MLX 2-bit   Qwen2.5-7B-Instruct


In [8]:
print_results_table(rows)


  SUMMARY
Model                     Variant      GSM8K %       PPL      Tok/s    TTFT ms    TPOT ms     Lat ms   Weight MB   Runtime MB     Peak MB
-----------------------------------------------------------------------------------------------------------------------------------------
Gemma-2-9B-it             MLX 4-bit       90.0      6.81      37.40     461.39      23.72    3877.15      4958.5        262.7      5221.2
Llama-3.1-8B-Instruct     MLX 4-bit       80.0      6.23      50.04     381.27      18.12    3904.57      4308.1        251.5      4559.6
Qwen2.5-7B-Instruct       FP16            80.0      5.08      16.72     462.54      58.00   13356.39     14525.6        126.4     14652.0
Qwen2.5-7B-Instruct       MLX 4-bit       80.0      5.61      53.11     373.14      17.38    4581.19      4088.9        242.6      4331.5
Qwen2.5-7B-Instruct       MLX 2-bit        0.0  13421.87      76.08     370.31      12.45    6729.85      2273.3        310.6      2583.8


In [9]:
# Write the union-of-keys CSV next to this notebook.
out_path = dump_results_table(str(Path.cwd()), rows, filename="qwen_summary.csv")
print("wrote:", out_path)

wrote summary csv: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/qwen_summary.csv (5 rows)
wrote: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/qwen_summary.csv


In [10]:
# Optional: preview as a pandas DataFrame for interactive filtering/sorting.
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    front = ["name", "variant", "gsm8k_accuracy", "perplexity", "throughput_tok_s", "ttft_mean_ms", "tpot_mean_ms", "weight_mem_mb", "peak_mem_mb"]
    df = df[front + [c for c in df.columns if c not in front]]
    display(df)
except ImportError:
    print("pandas not installed — skipping DataFrame preview")

,name,variant,gsm8k_accuracy,perplexity,throughput_tok_s,ttft_mean_ms,tpot_mean_ms,weight_mem_mb,peak_mem_mb,ttft_std_ms,...,latency_std_ms,tpot_std_ms,tpot_skipped,num_samples,total_tokens,prompt_len_mean,output_len_mean,gsm8k_correct,gsm8k_total,runtime_mem_mb
0,Gemma-2-9B-it,MLX 4-bit,0.9,6.806845,37.398566,461.393379,23.721875,4958.467781,5221.217199,49.067250,...,886.398517,0.027574,0,10,1450,74.7,145.0,9,10,262.749418
1,Llama-3.1-8B-Instruct,MLX 4-bit,0.8,6.227768,50.043874,381.274633,18.121384,4308.134285,4559.594440,39.655226,...,935.522136,0.078368,0,10,1954,98.7,195.4,8,10,251.460155
2,Qwen2.5-7B-Instruct,FP16,0.8,5.082452,16.718592,462.541587,57.996049,14525.635750,14651.995644,40.437159,...,2854.410321,0.253416,0,10,2233,95.0,223.3,8,10,126.359894
3,Qwen2.5-7B-Instruct,MLX 4-bit,0.8,5.609574,53.108465,373.142746,17.377552,4088.885750,4331.511406,44.266647,...,986.750059,0.089117,0,10,2433,95.0,243.3,8,10,242.625656
4,Qwen2.5-7B-Instruct,MLX 2-bit,0.0,13421.872589,76.078992,370.310587,12.445277,2273.260750,2583.839531,42.550584,...,46.288688,0.065650,0,10,5120,95.0,512.0,0,10,310.578781
